# Profilazione del carico di rete regionale e delle interruzioni con PROC CHART

## Sintesi esecutiva

Un analista delle operazioni di rete usa PROC CHART per profilare un campione sintetico di letture dei contatori dei circuiti di alimentazione su quattro regioni di servizio e quattro fonti di generazione. Il notebook percorre grafici a barre verticali, a barre orizzontali, a blocchi e a torta per riassumere il mix di generazione, confrontare il carico medio e totale per regione, evidenziare il picco di domanda serale per ora e classificare i minuti di interruzione per fonte — il tipo di esplorazione rapida con output testuale che un team energia-e-utility esegue prima di impegnarsi in un report grafico. Il passo DATA richiede 1.200 righe; questo notebook è stato reso in modalità senza licenza, che limita il dataset di lavoro alle prime 100 letture, quindi ogni grafico qui sotto riassume quel campione di 100 righe.

## Fonti dei dati

| Dataset | Righe | Descrizione |
|---------|------|-------------|
| `grid_ops` | 100 (campione sintetico) | Una riga per lettura del contatore di circuito di alimentazione su una rete regionale, generata inline con `call streaminit(20260531)` e `rand()`. Il ciclo del passo DATA richiede 1.200 letture, ma la modalità senza licenza limita il dataset salvato a 100 osservazioni, quindi i grafici qui sotto descrivono quelle 100 righe. |

# Profilazione del carico di rete regionale e delle interruzioni con PROC CHART

PROC CHART produce grafici a barre, a blocchi e a torta basati su caratteri direttamente nel listing — senza bisogno di un dispositivo ODS Graphics. Questo lo rende uno strumento di esplorazione rapida di primo passaggio per un team di operazioni di rete che vuole *vedere* la forma dei propri dati di carico e affidabilità prima di costruire visualizzazioni curate con GCHART o SGPLOT.

In questo notebook:

1. Generiamo un mese sintetico di letture dei contatori di circuito per un operatore di rete regionale.
2. Rappresentiamo il **mix di generazione** (quota di letture per fonte).
3. Confrontiamo il **carico medio e totale erogato** tra le regioni di servizio.
4. Evidenziamo il **picco di domanda serale** per ora del giorno.
5. Usiamo un **grafico a blocchi** per incrociare la regione con la fonte di generazione.
6. Classifichiamo i **minuti di interruzione** per fonte per individuare l'alimentazione meno affidabile.

Ogni istruzione e opzione qui sotto è sintassi standard PROC CHART di SAS 9.4.

## Passo 1 — Generare i dati sintetici delle operazioni

Il passo DATA qui sotto fabbrica letture dei contatori in un ciclo di 1.200 iterazioni. A ogni lettura viene assegnata una regione di servizio, una fonte di generazione (il Gas domina, con Eolico, Solare e Idroelettrico a comporre il resto) e un'ora del giorno. Il carico è più alto nella finestra serale 17:00–21:00 (e contrassegniamo quelle letture come picco), e traiamo i minuti di interruzione da una distribuzione di Poisson. `call streaminit` fissa il seme casuale così che i dati siano riproducibili.

La NOTE nel log mostra il risultato pratico: questa esecuzione è in modalità senza licenza, quindi il dataset salvato `grid_ops` è limitato alle prime 100 di quelle letture (100 righe, 7 colonne). Ogni grafico che segue riassume quel campione di 100 righe, e ogni log di PROC CHART conferma di aver letto 100 righe.

In [1]:
/* Synthetic feeder-circuit operations for a regional grid operator */
DATI grid_ops;
    CHIAMARE streaminit(20260531);
    LUNGHEZZA region $12 source $9 peak_flag $3;
    VETTORE regions[4] $12 _temporary_
        ('North','South','East','West');
    FARE meter_id = 1 FINO_A 1200;
        r = ceil(4 * rand('uniform'));
        region = regions[r];
        u = rand('uniform');
        SE_COND u < 0.42 ALLORA source = 'Gas';
        ALTRIMENTI SE_COND u < 0.70 ALLORA source = 'Wind';
        ALTRIMENTI SE_COND u < 0.88 ALLORA source = 'Solar';
        ALTRIMENTI source = 'Hydro';
        hour = floor(24 * rand('uniform'));
        BASE = 18 + 5 * (region = 'North') + 3 * (region = 'West');
        SE_COND hour >= 17 E_LOG hour <= 21 ALLORA FARE;
            load_mwh = BASE + 12 + 6 * rand('normal');
            peak_flag = 'Yes';
        FINE;
        ALTRIMENTI FARE;
            load_mwh = BASE + 4 * rand('normal');
            peak_flag = 'No';
        FINE;
        SE_COND load_mwh < 0 ALLORA load_mwh = 0;
        outage_min = rand('poisson', 2.5);
        USCITA;
    FINE;
    RIMUOVERE r u BASE;
ESEGUIRE;

NOTE: DATA grid_ops

NOTE: Unlicensed mode - output limited to 100 observations.

NOTE: Wrote grid_ops (100 rows, 7 columns).
NOTE: DATA elapsed:
  wall  0.01 seconds
  cpu   0.01 seconds


## Passo 2 — Mix di generazione

Quale quota di letture rappresenta ciascuna fonte di generazione? Un grafico a barre verticali con `TYPE=PERCENT` risponde direttamente: le altezze delle barre sono la percentuale di tutte le osservazioni che ricadono in ciascuna categoria di fonte. Poiché `source` è una variabile carattere, PROC CHART tratta automaticamente i suoi valori come categorie discrete.

In [2]:
PROCEDURA chart DATI=grid_ops;
    VBAR source / type=PERCENT;
ESEGUIRE;


Percent of SOURCE

         |              *****       
         |              *****       
   40.00 +              *****       
         |              *****       
         |              *****       
   30.00 +              *****       
         |       *****  *****       
         |       *****  *****       
   20.00 +       *****  *****       
         |       *****  *****       
         |*****  *****  *****       
         |*****  *****  *****  *****
   10.00 +*****  *****  *****  *****
         |*****  *****  *****  *****
         |*****  *****  *****  *****
         |                          
         +--------------------------+
          Hydro  Wind    Gas   Solar
                    SOURCE


NOTE: PROC CHART data=grid_ops

NOTE: 100 rows read from dataset 'grid_ops'.


## Passo 3 — Carico medio erogato per regione

Ora passiamo dal conteggio al riepilogo di una misurazione. Nominare `load_mwh` in `SUMVAR=` con `TYPE=MEAN` rende la lunghezza della barra il carico medio per regione, e richiediamo esplicitamente le colonne statistiche: `MEAN` stampa la media accanto a ogni barra e `CFREQ` aggiunge una colonna di frequenza cumulata. Il Nord porta il carico medio erogato più alto (media circa 28,0 MWh), coerentemente con lo scostamento regionale incorporato nei dati, mentre il Sud è il più basso (circa 19,8 MWh).

Passiamo anche `ASCENDING`, con l'intenzione di ordinare le barre dalla media più bassa alla più alta. Si noti nell'output che le barre **non** vengono riordinate — appaiono in ordine di categoria (Ovest, Sud, Est, Nord), con medie 24,2, 19,8, 21,7, 28,0 che non vanno dalla più corta alla più lunga. Il riordino delle barre secondo la statistica del grafico non è ancora implementato in questa versione di PROC CHART, quindi `ASCENDING`/`DESCENDING` sono accettati ma attualmente non hanno effetto; leggere la classifica dalla colonna `Mean` stampata.

In [3]:
PROCEDURA chart DATI=grid_ops;
    HBAR region / SUMVAR=load_mwh type=mean
                  cfreq mean ascending;
ESEGUIRE;


Mean of REGION

REGION                                                  Mean           N     Percent
                                                                                    
  West  **********************************             24.17       24.17       23.00
 South  ****************************                   19.80       43.97       21.00
  East  *******************************                21.72       65.69       29.00
 North  ****************************************       28.03       93.72       27.00


NOTE: PROC CHART data=grid_ops

NOTE: 100 rows read from dataset 'grid_ops'.


## Passo 4 — Carico totale per regione

Qui `TYPE=SUM` rende ogni barra il carico erogato *totale* della regione anziché la media, quindi le barre più alte segnano le regioni che erogano più energia aggregata sulle letture campionate. Il Nord guida di nuovo il carico totale erogato.

L'istruzione richiede anche `SUBGROUP=peak_flag`, chiedendo a PROC CHART di suddividere ciascuna barra a seconda che la lettura ricada nella finestra di picco serale. In SAS questo segmenta ogni barra con un glifo distinto per livello di sottogruppo e stampa una legenda. Questa versione non rende ancora la segmentazione per sottogruppo (una lacuna di capacità documentata), quindi le barre risultano come sequenze piene di `*` senza ripartizione per segmento — i *totali* delle barre sono corretti, ma la suddivisione picco-vs-fuori-picco non è mostrata qui. Per vedere quanto carico ricade nelle ore di picco, usare la vista per ora del giorno nel Passo 5.

In [4]:
PROCEDURA chart DATI=grid_ops;
    VBAR region / SUMVAR=load_mwh type=sum
                  SUBGROUP=peak_flag;
ESEGUIRE;


Sum of REGION

         |                     *****
         |                     *****
         |                     *****
     600 +              *****  *****
         |*****         *****  *****
         |*****         *****  *****
         |*****         *****  *****
     400 +*****  *****  *****  *****
         |*****  *****  *****  *****
         |*****  *****  *****  *****
         |*****  *****  *****  *****
     200 +*****  *****  *****  *****
         |*****  *****  *****  *****
         |*****  *****  *****  *****
         |*****  *****  *****  *****
         |                          
         +--------------------------+
          West   South  East   North
                    REGION


NOTE: PROC CHART data=grid_ops

NOTE: 100 rows read from dataset 'grid_ops'.


## Passo 5 — Andamento del carico durante la giornata

`hour` è continua, quindi fissiamo noi stessi la suddivisione in classi con un elenco `MIDPOINTS=` esplicito centrato su intervalli di 4 ore (2, 6, 10, 14, 18, 22). Ogni barra mostra il carico medio erogato per le letture vicine a quell'ora. La barra centrata su 18 dovrebbe risaltare — è il picco di domanda serale che abbiamo incorporato nei dati.

In [5]:
PROCEDURA chart DATI=grid_ops;
    VBAR hour / SUMVAR=load_mwh type=mean
                midpoints=2 6 10 14 18 22;
ESEGUIRE;


Mean of HOUR

         |                            *****       
         |                            *****       
   30.00 +                            *****       
         |                            *****  *****
         |                            *****  *****
         |              *****         *****  *****
   20.00 +*****  *****  *****  *****  *****  *****
         |*****  *****  *****  *****  *****  *****
         |*****  *****  *****  *****  *****  *****
         |*****  *****  *****  *****  *****  *****
         |*****  *****  *****  *****  *****  *****
   10.00 +*****  *****  *****  *****  *****  *****
         |*****  *****  *****  *****  *****  *****
         |*****  *****  *****  *****  *****  *****
         |*****  *****  *****  *****  *****  *****
         |                                        
         +----------------------------------------+
            2      6     10     14     18     22  
                            HOUR


NOTE: PROC CHART data=grid_ops


## Passo 6 — Regione per fonte di generazione (grafico a blocchi)

Un grafico BLOCK rende una piccola tabella a doppia entrata come una griglia di blocchi. Con `GROUP=source` e `SUMVAR=load_mwh / TYPE=MEAN`, il grafico incrocia ciascuna regione con la fonte di generazione che la serve, con l'altezza del blocco proporzionale al carico medio — un modo compatto per individuare quali combinazioni regione/fonte portano il carico medio più pesante.

In [6]:
PROCEDURA chart DATI=grid_ops;
    BLOCK region / SUMVAR=load_mwh type=mean
                   GROUP=source;
ESEGUIRE;


Block chart of Mean of REGION by SOURCE

                          /####\
  /####\                  /####\
  /####\          /####\  /####\
  /####\  /####\  /####\  /####\
  /####\  /####\  /####\  /####\
  /####\  /####\  /####\  /####\
  /####\  /####\  /####\  /####\
  /####\  /####\  /####\  /####\
  /####\  /####\  /####\  /####\
  /####\  /####\  /####\  /####\
  +----+  +----+  +----+  +----+
   West   South    East   North 
             REGION


NOTE: PROC CHART data=grid_ops

NOTE: 100 rows read from dataset 'grid_ops'.


## Passo 7 — Mix di generazione come grafico a torta

Le stesse informazioni sulla quota per fonte del Passo 2, disegnate come una torta. PIE con `TYPE=PERCENT` dimensiona ogni fetta in base alla sua percentuale sul totale delle letture e stampa una legenda delle percentuali delle fette accanto alla figura.

In [7]:
PROCEDURA chart DATI=grid_ops;
    PIE source / type=PERCENT;
ESEGUIRE;


Pie chart of SOURCE

          SOURCE     Percent   Percent  Slice
----------------  ----------  --------  ------------------------------
           Hydro       14.00     14.0%  ####
            Wind       28.00     28.0%  ********
             Gas       45.00     45.0%  ++++++++++++++
           Solar       13.00     13.0%  OOOO


NOTE: PROC CHART data=grid_ops

NOTE: 100 rows read from dataset 'grid_ops'.


## Passo 8 — Minuti di interruzione per fonte

Infine classifichiamo l'affidabilità. `SUMVAR=outage_min` con `TYPE=SUM` somma i minuti di interruzione per fonte. Passiamo `DESCENDING` per tentare di portare in cima la fonte con le prestazioni peggiori, ma come nel Passo 3 le barre non vengono riordinate — vengono stampate in ordine di categoria (Idroelettrico, Eolico, Gas, Solare) e il riordino delle barre non è ancora implementato. Leggere la classifica dalla colonna `Sum` stampata: il Gas rappresenta il maggior numero totale di minuti di interruzione in questo campione (122), ben davanti a Eolico (64), Solare (43) e Idroelettrico (38). Ciò rispecchia il mix di generazione — il Gas serve il maggior numero di letture, quindi accumula complessivamente più minuti di interruzione.

In [8]:
PROCEDURA chart DATI=grid_ops;
    HBAR source / SUMVAR=outage_min type=sum DISCENDENTE;
ESEGUIRE;


Sum of SOURCE

SOURCE                                                   Sum        Cum.     Percent
                                                                     Sum            
 Hydro  ************                                      38          38       14.00
  Wind  *********************                             64         102       28.00
   Gas  ****************************************         122         224       45.00
 Solar  **************                                    43         267       13.00


NOTE: PROC CHART data=grid_ops

NOTE: 100 rows read from dataset 'grid_ops'.


## Interpretazione dei risultati

Leggere i grafici insieme offre al team delle operazioni un quadro situazionale rapido (sul campione di 100 righe catturato da questa esecuzione):

- **Mix di generazione (Passi 2 e 7).** Il Gas porta la quota più grande di letture (circa il 45%), con l'Eolico secondo (circa il 28%) e Idroelettrico e Solare a seguire (circa il 14% e il 13%) — la barra verticale e la torta raccontano la stessa storia in due modi, un utile controllo di coerenza.
- **Carico per regione (Passi 3 e 4).** L'HBAR del carico medio mostra il Nord con il carico medio erogato più alto (media ~28 MWh) e il Sud il più basso (~20 MWh), coerentemente con lo scostamento regionale incorporato nei dati. Il grafico SUM conferma che il Nord eroga anche la maggior energia totale.
- **Andamento del carico giornaliero (Passo 5).** La barra del punto medio centrata sull'ora 18 è chiaramente la più alta, confermando il picco di domanda 17:00–21:00 che abbiamo incorporato nei dati — esattamente dove una utility concentrerebbe la risposta alla domanda e la pianificazione della capacità.
- **Affidabilità (Passo 8).** Sommando i minuti di interruzione per fonte emerge il Gas come il maggior contributore di fermo in questo campione (122 minuti), il naturale obiettivo successivo per la revisione della manutenzione — anche se ciò riflette per lo più il fatto che il Gas serve il maggior numero di letture.

Due opzioni di visualizzazione usate qui — il riordino delle barre `ASCENDING`/`DESCENDING` (Passi 3 e 8) e la segmentazione delle barre `SUBGROUP=` (Passo 4) — sono accettate dal parser ma non ancora rese da questa versione di PROC CHART, quindi le classifiche e le suddivisioni si leggono dalle colonne statistiche stampate anziché dall'ordine o dall'ombreggiatura delle barre.

PROC CHART produce solo output a caratteri, quindi per visuali pronte per il consiglio direttivo il team sposterebbe queste stesse viste su PROC GCHART o PROC SGPLOT. Ma come primo passaggio a configurazione zero su un'estrazione appena effettuata, questi grafici rispondono alle domande operative — mix, entità e tempistica — in pochi secondi.